# Task 4 - Evaluation

Evaluate the best GRU/Luong checkpoint on UQA validation and Wiki-UQA. This notebook reports BLEU-4, ROUGE-L, perplexity, and `<unk>` rate for greedy and beam-search decoding, then saves 50 fixed validation samples.

In [ ]:
!pip install sacrebleu rouge-score

import csv
import math
import random
from collections import defaultdict
from pathlib import Path

import sacrebleu
import sentencepiece as spm
import torch
import torch.nn as nn
from google.colab import drive
from rouge_score import rouge_scorer
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PAD, UNK, BOS, EOS = 0, 1, 2, 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', DEVICE)

In [ ]:
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/GenAI-Dataset (1)')
MODEL_DIR = DRIVE_DIR / 'model'
RESULTS_DIR = DRIVE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DRIVE_DIR / 'train.tsv'
VALID_PATH = DRIVE_DIR / 'valid.tsv'
WIKI_PATH = DRIVE_DIR / 'wiki_test.tsv'
SP_MODEL_PATH = DRIVE_DIR / 'ur_sp.model'
CHECKPOINT_PATH = MODEL_DIR / 'best_model.pt'

for path in [TRAIN_PATH, VALID_PATH, WIKI_PATH, SP_MODEL_PATH, CHECKPOINT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')

In [ ]:
class Seq2SeqDataset(Dataset):
    def __init__(self, tsv_path, sp_model_path):
        self.sp = spm.SentencePieceProcessor(model_file=str(sp_model_path))
        with open(tsv_path, encoding='utf-8') as file:
            rows = csv.reader(file, delimiter='\t', quoting=csv.QUOTE_NONE, escapechar='\\')
            self.pairs = list(dict.fromkeys((row[0], row[1]) for row in rows if len(row) == 2))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, index):
        source, target = self.pairs[index]
        source_ids = self.sp.encode(source, out_type=int)
        target_ids = [BOS] + self.sp.encode(target, out_type=int) + [EOS]
        return torch.tensor(source_ids), torch.tensor(target_ids)

def collate_fn(batch):
    sources, targets = zip(*batch)
    lengths = torch.tensor([len(x) for x in sources], dtype=torch.long)
    return (pad_sequence(sources, batch_first=True, padding_value=PAD), lengths,
            pad_sequence(targets, batch_first=True, padding_value=PAD))

valid_dataset = Seq2SeqDataset(VALID_PATH, SP_MODEL_PATH)
wiki_dataset = Seq2SeqDataset(WIKI_PATH, SP_MODEL_PATH)
VOCAB_SIZE = valid_dataset.sp.get_piece_size()
print(f'Validation: {len(valid_dataset):,}; Wiki-UQA: {len(wiki_dataset):,}; vocab: {VOCAB_SIZE:,}')

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self, hidden_size=512):
        super().__init__()
        self.project = nn.Linear(hidden_size * 2, hidden_size, bias=False)

    def forward(self, encoder_outputs, decoder_outputs, source_mask):
        scores = torch.bmm(decoder_outputs, self.project(encoder_outputs).transpose(1, 2))
        weights = torch.softmax(scores.masked_fill(~source_mask.unsqueeze(1), -1e9), dim=2)
        context = torch.bmm(weights, encoder_outputs)
        return context, weights

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size=256, hidden_size=512, layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embedding_size, hidden_size, layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.hidden_bridge = nn.Linear(hidden_size * 2, hidden_size)
        self.layers = layers

    def forward(self, source, lengths):
        packed = pack_padded_sequence(self.dropout(self.embedding(source)), lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=source.size(1))
        hidden = hidden.view(self.layers, 2, source.size(0), -1)
        hidden = torch.tanh(self.hidden_bridge(torch.cat([hidden[:, 0], hidden[:, 1]], dim=-1)))
        return outputs, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_size=256, hidden_size=512, layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.attention = LuongAttention(hidden_size)
        self.gru = nn.GRU(embedding_size, hidden_size, layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_size * 3, vocab_size)

    def forward(self, tokens, hidden, encoder_outputs, source_mask):
        outputs, hidden = self.gru(self.dropout(self.embedding(tokens)), hidden)
        context, weights = self.attention(encoder_outputs, outputs, source_mask)
        return self.output(torch.cat([outputs, context], dim=-1)), hidden, weights

    def step(self, token, hidden, encoder_outputs, source_mask):
        logits, hidden, weights = self.forward(token.unsqueeze(1), hidden, encoder_outputs, source_mask)
        return logits.squeeze(1), hidden, weights.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = Encoder(vocab_size)
        self.decoder = Decoder(vocab_size)

    def forward(self, source, lengths, target):
        encoder_outputs, hidden = self.encoder(source, lengths)
        logits, _, _ = self.decoder(target[:, :-1], hidden, encoder_outputs, source.ne(PAD))
        return logits

model = Seq2Seq(VOCAB_SIZE).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()
sp = valid_dataset.sp
print('Best checkpoint loaded.')

In [ ]:
MAX_LENGTH, BEAM_SIZE = 60, 3
BLOCKED_IDS = [PAD, BOS, sp.piece_to_id('<ans>'), sp.piece_to_id('</ans>')]

def mask_invalid(logits):
    logits[:, BLOCKED_IDS] = -float('inf')
    return logits

def beam_score(beam):
    length = max(1, len(beam[0]))
    return beam[1] / (((5 + length) / 6) ** 0.6)

@torch.no_grad()
def encode_source(source):
    ids = torch.tensor([sp.encode(source, out_type=int)], device=DEVICE)
    outputs, hidden = model.encoder(ids, torch.tensor([ids.size(1)]))
    return outputs, hidden, ids.ne(PAD)

@torch.no_grad()
def greedy_decode(source):
    outputs, hidden, mask = encode_source(source)
    token, ids, ended = torch.tensor([BOS], device=DEVICE), [], False
    for _ in range(MAX_LENGTH):
        logits, hidden, _ = model.decoder.step(token, hidden, outputs, mask)
        token = mask_invalid(logits).argmax(dim=-1)
        if token.item() == EOS:
            ended = True
            break
        ids.append(token.item())
    return sp.decode(ids), ids, ended

@torch.no_grad()
def beam_decode(source):
    outputs, hidden, mask = encode_source(source)
    beams = [([], 0.0, torch.tensor([BOS], device=DEVICE), hidden, False)]
    for _ in range(MAX_LENGTH):
        candidates = []
        for ids, score, token, state, ended in beams:
            if ended:
                candidates.append((ids, score, token, state, ended)); continue
            logits, next_state, _ = model.decoder.step(token, state, outputs, mask)
            scores, tokens = torch.log_softmax(mask_invalid(logits), dim=-1).topk(BEAM_SIZE, dim=-1)
            for next_score, next_token in zip(scores[0], tokens[0]):
                token_id = next_token.item()
                candidates.append((ids + [token_id], score + next_score.item(), next_token.unsqueeze(0),
                                   next_state.clone(), token_id == EOS))
        beams = sorted(candidates, key=beam_score, reverse=True)[:BEAM_SIZE]
        if all(beam[4] for beam in beams): break
    finished = [beam for beam in beams if beam[4]]
    best = max(finished or beams, key=beam_score)
    ids = best[0]
    return sp.decode(ids[:ids.index(EOS)] if EOS in ids else ids), [x for x in ids if x != EOS], best[4]

In [ ]:
class UrduWhitespaceTokenizer:
    def tokenize(self, text):
        return text.split()

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False, tokenizer=UrduWhitespaceTokenizer())

def perplexity(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
    total_loss, tokens = 0.0, 0
    with torch.no_grad():
        for source, lengths, target in loader:
            source, target = source.to(DEVICE), target.to(DEVICE)
            logits = model(source, lengths, target)
            total_loss += loss_fn(logits.reshape(-1, VOCAB_SIZE), target[:, 1:].reshape(-1)).item()
            tokens += target[:, 1:].ne(PAD).sum().item()
    return math.exp(total_loss / tokens)

def group_references(dataset):
    grouped = defaultdict(list)
    for source, reference in dataset.pairs:
        if reference not in grouped[source]:
            grouped[source].append(reference)
    return list(grouped.items())

def metrics(hypotheses, references, generated_ids):
    max_refs = max(len(refs) for refs in references)
    reference_streams = [[refs[i] if i < len(refs) else None for refs in references]
                         for i in range(max_refs)]
    bleu = sacrebleu.corpus_bleu(hypotheses, reference_streams).score
    rouge_l = sum(max(rouge.score(ref, hyp)['rougeL'].fmeasure for ref in refs)
                  for refs, hyp in zip(references, hypotheses)) / len(references)
    tokens = sum(len(ids) for ids in generated_ids)
    unk_rate = sum(token == UNK for ids in generated_ids for token in ids) / max(1, tokens)
    return bleu, rouge_l, unk_rate

def evaluate(dataset, decoder, label):
    sources, references = zip(*group_references(dataset))
    hypotheses, ids, ended = [], [], []
    for source in tqdm(sources, desc=label):
        prediction, prediction_ids, reached_eos = decoder(source)
        hypotheses.append(prediction)
        ids.append(prediction_ids)
        ended.append(reached_eos)
    return list(sources), hypotheses, list(references), ids, ended

results = []
all_outputs = {}
for split_name, dataset in [('UQA valid', valid_dataset), ('Wiki-UQA', wiki_dataset)]:
    split_ppl = perplexity(dataset)
    for decoding, decoder in [('greedy', greedy_decode), ('beam (k=3)', beam_decode)]:
        sources, hypotheses, references, ids, ended = evaluate(dataset, decoder, f'{split_name}: {decoding}')
        bleu, rouge_l, unk_rate = metrics(hypotheses, references, ids)
        results.append({'split': split_name, 'decoding': decoding, 'BLEU-4': bleu, 'ROUGE-L': rouge_l,
                        'PPL': split_ppl, 'unk_rate': unk_rate, 'EOS_rate': sum(ended) / len(ended),
                        'truncation_rate': 1 - sum(ended) / len(ended),
                        'avg_length': sum(map(len, ids)) / len(ids)})
        all_outputs[(split_name, decoding)] = sources, references, hypotheses

for row in results:
    print(f"{row['split']:9} | {row['decoding']:10} | BLEU-4 {row['BLEU-4']:.2f} | ROUGE-L {row['ROUGE-L']:.4f} | PPL {row['PPL']:.2f} | unk {row['unk_rate']:.2%} | EOS {row['EOS_rate']:.2%} | truncated {row['truncation_rate']:.2%}")

In [ ]:
with open(RESULTS_DIR / 'metrics.tsv', 'w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=['split', 'decoding', 'BLEU-4', 'ROUGE-L', 'PPL', 'unk_rate',
                                                   'EOS_rate', 'truncation_rate', 'avg_length'], delimiter='\t')
    writer.writeheader(); writer.writerows(results)

sources, references, greedy = all_outputs[('UQA valid', 'greedy')]
_, _, beam = all_outputs[('UQA valid', 'beam (k=3)')]
sample_indices = random.Random(42).sample(range(len(sources)), 50)
with open(RESULTS_DIR / 'samples.tsv', 'w', encoding='utf-8', newline='') as file:
    writer = csv.writer(file, delimiter='\t', quoting=csv.QUOTE_NONE, escapechar='\\')
    writer.writerow(['source', 'reference', 'greedy', 'beam'])
    for index in sample_indices:
        writer.writerow([sources[index], ' ||| '.join(references[index]), greedy[index], beam[index]])

print('Saved:', RESULTS_DIR / 'metrics.tsv')
print('Saved:', RESULTS_DIR / 'samples.tsv')